In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, roc_auc_score

from src.features.preprocessing import load_model_data
from src.evaluation import splits, leakage
from src.evaluation.make_splits import load_split, SPLITS

SPLIT_LIST = list(SPLITS)

In [9]:
# ===== Settings: change these, then Restart & Run All =====
SPLIT = "10km"            # "10km", "10km_subbasin", "7.5km_subbasin", "5km_subbasin"
USE_SUBBASIN = False      # True: subbasin features replace elevation
MODEL_SEED = 0
THRESHOLD_MG_L = 10

In [10]:
X, y, meta = load_model_data(subbasin=USE_SUBBASIN)
FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=SPLIT)
print(f"split={SPLIT} | subbasin features={USE_SUBBASIN} | {len(FEATURES)} features")
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))


split=10km | subbasin features=False | 50 features
{0: 1035, 1: 972, 2: 1143, 3: 1041, 4: 1089} | holdout rows: 1089


In [11]:
from src.features.transformers import add_subbasin

if "subbasin" not in data.columns:
    data = add_subbasin(data)          # for reporting only; not added to FEATURES

print(data.groupby("fold")["subbasin"].value_counts(normalize=True).unstack().round(2))


subbasin  180/400 FOOT AQUIFER  ATASCADERO AREA  EAST SIDE AQUIFER  \
fold                                                                 
0                         0.08             0.10               0.33   
1                         0.41             0.00               0.06   
2                         0.31             0.03               0.23   
3                         0.00             0.10               0.00   
4                         0.13             0.00               0.03   

subbasin  FOREBAY AQUIFER  LANGLEY AREA  MONTEREY  PASO ROBLES AREA  SEASIDE  \
fold                                                                           
0                    0.20           0.0      0.00              0.20      0.0   
1                    0.23           0.0      0.02              0.02      0.0   
2                    0.05           0.0      0.00              0.24      0.0   
3                    0.45           0.0      0.00              0.18      0.0   
4                    0.50    

In [12]:
# Set the training folds and the holdout

train_val = data[~data["holdout"]].reset_index(drop=True)       # Training set with 4 folds


X_tv = train_val[FEATURES]
y_tv = train_val["log_nitrate"]

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 1035, 1: 972, 2: 1143, 3: 1041}


Here you can begin training your model.

In [13]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.base import clone
import xgboost

In [14]:
def make_model():
    """Replace with your model. A new, untrained model is created for every fold."""
    return DummyRegressor(strategy="mean")

xy = leakage.projected_xy(train_val, "gm_latitude", "gm_longitude")
oof_pred = np.full(len(train_val), np.nan)        # model prediction, out of fold
mean_pred = np.full(len(train_val), np.nan)       # baseline: training-fold mean
nearest_m = np.full(len(train_val), np.nan)       # distance to nearest training well
fold_scores = []

for train, test in splits.iter_folds(train_val["fold"]):
    model = make_model()
    model.fit(X_tv[train], y_tv[train])
    oof_pred[test] = model.predict(X_tv[test])
    mean_pred[test] = y_tv[train].mean()
    nearest_m[test] = leakage.nearest_training_distance(xy, train, test)

    fold_scores.append({
        "fold": int(train_val.loc[test, "fold"].iloc[0]),
        "n": int(test.sum()),
        "rmse_log": root_mean_squared_error(y_tv[test], oof_pred[test]),
        "rmse_mean": root_mean_squared_error(y_tv[test], mean_pred[test]),
    })

exceeds = np.expm1(y_tv) > THRESHOLD_MG_L
print(pd.DataFrame(fold_scores).round(3).to_string(index=False))
print(f"Overall RMSE (log): {root_mean_squared_error(y_tv, oof_pred):.3f} "
      f"| R²: {r2_score(y_tv, oof_pred):.3f} "
      f"| AUC (>{THRESHOLD_MG_L} mg/L): {roc_auc_score(exceeds, oof_pred):.3f}")


 fold    n  rmse_log  rmse_mean
    0 1035     1.227      1.227
    1  972     1.183      1.183
    2 1143     1.222      1.222
    3 1041     1.314      1.314
Overall RMSE (log): 1.238 | R²: -0.005 | AUC (>10 mg/L): 0.497


# Nested CV Example

In [15]:
FEATURES

['gm_well_category',
 'soil_slopegradwta',
 'soil_aws0150wta',
 'soil_hydgrpdcd_pct_a',
 'soil_hydgrpdcd_pct_b',
 'soil_hydgrpdcd_pct_c',
 'elevation_m',
 'topographic_position',
 'pct_veg_berry_500m',
 'pct_vineyard_500m',
 'pct_orchard_500m',
 'pct_grain_field_pasture_500m',
 'pct_idle_fallow_500m',
 'pct_urban_500m',
 'pct_cropland_500m',
 'crop_cycles_500m',
 'pct_multicrop_500m',
 'outside_dwr_polygons_pct_500m',
 'pct_orchard_1000m',
 'pct_grain_field_pasture_1000m',
 'pct_idle_fallow_1000m',
 'pct_urban_1000m',
 'outside_dwr_polygons_pct_1000m',
 'delta_veg_berry_pp_500m',
 'delta_vineyard_pp_500m',
 'delta_orchard_pp_500m',
 'delta_grain_field_pasture_pp_500m',
 'delta_idle_fallow_pp_500m',
 'delta_cropland_pp_500m',
 'delta_multicrop_pp_500m',
 'delta_dwr_polygon_footprint_pp_500m',
 'delta_veg_berry_pp_1000m',
 'delta_vineyard_pp_1000m',
 'delta_orchard_pp_1000m',
 'delta_grain_field_pasture_pp_1000m',
 'delta_idle_fallow_pp_1000m',
 'delta_urban_pp_1000m',
 'delta_cropland_p

In [16]:
SUBFOLD_SEED = 77
MODEL_SEED = 99
N_SUBFOLDS = 4
THRESHOLD_MG_L = 10

In [17]:
from src.models import random_forest
# "data" comes from load_split (see earlier template)
# this function creates the subfolds

development, X_dev, y_dev = random_forest.make_inner_subfolds(data, FEATURES, N_SUBFOLDS = N_SUBFOLDS, THRESHOLD_MG_L = THRESHOLD_MG_L, SEED = SUBFOLD_SEED)

subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471


In [18]:
param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [50, 100, 200, 500],
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 5, 20],
    "subsample": [0.7, 1.0],
}


In [19]:
# # customize this function to your estimator. Leave all other fields as is. 
# from sklearn.model_selection import GridSearchCV
# import xgboost

# def make_search_brt(inner_cv, param_grid, SEED):
#     return GridSearchCV(
#         estimator=xgboost.XGBRegressor(
#             objective="reg:squarederror",
#             random_state=SEED,
#             n_jobs=1,
#         ),
#         param_grid=param_grid,
#         scoring="neg_root_mean_squared_error",
#         cv=inner_cv,
#         refit=True,
#         n_jobs=-1,
#     )


In [20]:
from sklearn.model_selection import RandomizedSearchCV

def make_search_brt(inner_cv, param_grid, SEED):
    return RandomizedSearchCV(
        estimator=xgboost.XGBRegressor(objective="reg:squarederror", random_state=SEED, n_jobs=1),
        param_distributions=param_grid,
        n_iter=40,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        refit=True,
        random_state=SEED,
        n_jobs=-1,
    )


In [21]:
#call the function fit_model_with_nested_cv

outer_results, best_params_by_fold = (
random_forest.fit_model_with_nested_cv(
        development,
        X_dev,
        y_dev,
        param_grid,
        make_search_brt,
        N_SUBFOLDS=N_SUBFOLDS,
        THRESHOLD_MG_L=THRESHOLD_MG_L,
        SEED=MODEL_SEED,
        PRINT=True
    )
)

Nested CV results (outer and inner fold RMSE):
 outer_fold  rmse_log  best_inner_rmse_log  inner_fold_0_rmse_log  inner_fold_1_rmse_log  inner_fold_2_rmse_log  inner_fold_3_rmse_log
          0     1.040                1.062                  1.065                  1.078                  0.911                  1.196
          1     1.134                1.007                  0.990                  1.058                  0.745                  1.236
          2     1.102                1.007                  0.791                  0.974                  0.969                  1.296
          3     1.080                1.055                  1.058                  1.092                  0.871                  1.201

Best parameters per outer fold:
 outer_fold  subsample  n_estimators  min_child_weight  max_depth  learning_rate
          0        0.7            50                 1          3           0.05
          1        0.7            50                20          4           0.05
  

In [22]:
def find_best_hyperparameter(all_best_params, ignore=("outer_fold", "seed")):
    param_cols = [c for c in all_best_params.columns if c not in ignore]
    overall = (all_best_params
               .groupby(param_cols, dropna=False)
               .size()
               .reset_index(name="count")
               .sort_values("count", ascending=False))
    overall["frequency"] = overall["count"] / len(all_best_params)
    return overall


In [23]:
runs = []
for seed in [1, 2, 3]:
    outer_results, best_params_by_fold = random_forest.fit_model_with_nested_cv(
        development, X_dev, y_dev, param_grid, make_search_brt,
        N_SUBFOLDS=N_SUBFOLDS, THRESHOLD_MG_L=THRESHOLD_MG_L, SEED=seed, PRINT=False,
    )
    runs.append(best_params_by_fold.assign(seed=seed))

all_best_params = pd.concat(runs, ignore_index=True)       # 3 runs × 4 outer folds = 12 rows


In [24]:
frequency = find_best_hyperparameter(all_best_params)
frequency


,subsample,n_estimators,min_child_weight,max_depth,learning_rate,count,frequency
1,0.7,50,20,4,0.03,2,0.166667
3,0.7,100,20,3,0.03,2,0.166667
5,1.0,50,20,3,0.05,2,0.166667
4,1.0,50,1,4,0.05,2,0.166667
2,0.7,100,5,4,0.03,1,0.083333
0,0.7,50,1,4,0.03,1,0.083333
6,1.0,50,20,4,0.05,1,0.083333
7,1.0,100,5,4,0.03,1,0.083333


In [25]:
param_cols = [c for c in all_best_params.columns if c not in ("outer_fold", "seed")]
best_params = frequency[param_cols].iloc[[0]].to_dict("records")[0]
best_params


{'subsample': 0.7,
 'n_estimators': 50,
 'min_child_weight': 20,
 'max_depth': 4,
 'learning_rate': 0.03}

In [26]:
# Normal CV function (just outer folds)

model = xgboost.XGBRegressor(learning_rate=.1, max_depth=3, n_estimators=100)
random_forest.fit_model_with_normal_cv(development, X_dev, y_dev, model, THRESHOLD_MG_L = 10)

CV results (outer fold RMSE):
 outer_fold  rmse_log
          0     1.073
          1     1.299
          2     1.093
          3     1.060
Overall RMSE (log): 1.132
Accuracy of the >10 mg/L call: 0.695


In [27]:
def cv_summary(split, use_subbasin):
    X, y, meta = load_model_data(subbasin=use_subbasin)
    d = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=split)
    tv = d[~d["holdout"]].reset_index(drop=True)
    feats, yy = list(X.columns), tv["log_nitrate"]
    pred, base = np.full(len(tv), np.nan), np.full(len(tv), np.nan)
    for train, test in splits.iter_folds(tv["fold"]):
        pred[test] = make_model().fit(tv.loc[train, feats], yy[train]).predict(tv.loc[test, feats])
        base[test] = yy[train].mean()
    return {"split": split, "subbasin": use_subbasin,
            "rmse_log": root_mean_squared_error(yy, pred), "rmse_mean": root_mean_squared_error(yy, base),
            "r2": r2_score(yy, pred), "auc": roc_auc_score(np.expm1(yy) > THRESHOLD_MG_L, pred)}

pd.DataFrame([cv_summary(s, sb) for s in SPLIT_LIST
                                for sb in [False, True]]).round(3)


,split,subbasin,rmse_log,rmse_mean,r2,auc
0,10km,False,1.238,1.238,-0.005,0.497
1,10km,True,1.238,1.238,-0.005,0.497
2,5km_subbasin,False,1.210,1.210,-0.004,0.472
3,5km_subbasin,True,1.210,1.210,-0.004,0.472
4,7.5km_subbasin,False,1.215,1.215,-0.010,0.457
5,7.5km_subbasin,True,1.215,1.215,-0.010,0.457
6,10km_subbasin,False,1.210,1.210,-0.003,0.444
7,10km_subbasin,True,1.210,1.210,-0.003,0.444


# Final Evaluation

Save for when we have chosen our final model (along with any hyperparameters) to evaluate on the test set.

In [28]:
# ============================================================
# FINAL EVALUATION ON THE HOLD-OUT FOLD
# Run this ONCE, for the final chosen model only, after all
# model choices (features, hyperparameters) are finished.
# ============================================================
RUN_FINAL_EVALUATION = False

if RUN_FINAL_EVALUATION:
    holdout_df = data[data["holdout"]].reset_index(drop=True)
    final_model = make_model()
    final_model.fit(X_tv, y_tv)                                 # all four development folds
    holdout_pred = final_model.predict(holdout_df[FEATURES])
    y_ho = holdout_df["log_nitrate"]
    print(f"Hold-out ({SPLIT}) RMSE (log): {root_mean_squared_error(y_ho, holdout_pred):.3f} "
          f"| R²: {r2_score(y_ho, holdout_pred):.3f} "
          f"| AUC: {roc_auc_score(np.expm1(y_ho) > THRESHOLD_MG_L, holdout_pred):.3f}")
